# UMUD segment-then-measure (Kaggle GPU) &nbsp;?&nbsp; **version 2026-06-13.01**

> **How to know it's the current code:** cell 1 prints `downloaded pipeline VERSION: 2026-06-13.01` and will *error* if GitHub served a stale script. Each run also prints `##### UMUD pipeline VERSION 2026-06-13.01 #####` with the active flags at the top. If you don't see those, you're on old code ? re-run cell 1.

Trains the aponeurosis and fascicle U-Nets, predicts masks on the 309 test images, measures geometry (PCA-fit pennation + length-weighted median, TTA-denoised masks), applies the **per-family tick/ruler scale router** (`scale_ticks.py`) for muscle thickness on ~295/309 images, derives fascicle length from the measured geometry, and writes `submission_segmentation.csv`.

**How to run (no local setup needed):**
1. Open this notebook on Kaggle (New Notebook, then File > Import Notebook, upload this `.ipynb`).
2. Right panel > **Add Input** > add the competition *UMUD Challenge: Muscle Architecture in Ultrasound Data*.
3. Right panel > Settings: **Accelerator = GPU** (T4 or P100) and **Internet = On**.
4. **Run All**. The optional smoke cell (2 epochs) finishes in a few minutes; the full cell takes roughly 40-60 minutes.
5. Cell 6 self-checks that calibration ran (~295 scaled). Cell 7 gives download links: the submission, the calibration debug CSV, and the trained weights (`seg_apo.pt`, `seg_fasc.pt`).

**Note:** this competition is a CSV upload. The repo already contains a validated `results/submission_local.csv` (same pipeline on the existing saved weights, CPU). To just see the score without a retrain, upload that CSV directly. Run this notebook when you want fresh weights or a reproducible end-to-end run.

In [ ]:
# Dependencies and the latest pipeline scripts (needs Internet = On)
!pip install -q segmentation-models-pytorch albumentations
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/segment_then_measure.py -O segment_then_measure.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/tick_calibration.py -O tick_calibration.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/scale_ticks.py -O scale_ticks.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/subpixel_spacing.py -O subpixel_spacing.py
import os
for f in ('segment_then_measure.py', 'tick_calibration.py', 'scale_ticks.py', 'subpixel_spacing.py'):
    assert os.path.exists(f), f'{f} failed to download - check Internet = On'
# VERSION GATE: confirm we pulled the CURRENT scripts (GitHub raw can cache for a minute or two).
EXPECTED_VERSION = '2026-06-13.01'
got = next((ln.split('"')[1] for ln in open('segment_then_measure.py') if ln.startswith('PIPELINE_VERSION')), 'MISSING')
print(f'>>> downloaded pipeline VERSION: {got}   (expected {EXPECTED_VERSION}) <<<')
assert got == EXPECTED_VERSION, 'STALE script: GitHub is serving an older version. Wait 1-2 min and re-run THIS cell only.'
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
import os
# ===== training config for THIS run (edit, then Run All) =====
# The scale router + FL/MT geometry are INFERENCE-side and always on; they do NOT need retraining.
os.environ['UMUD_EPOCHS'] = '12'            # the standard that produced our current weights
os.environ['UMUD_IMG_SIZE'] = '384'         # try 512/640 for serious segmentation runs
os.environ['UMUD_MODEL_ARCH'] = 'unet'      # unet | unetplusplus | fpn | deeplabv3plus
os.environ['UMUD_MODEL_ENCODER'] = 'resnet34'
os.environ['UMUD_LOSS_MODE'] = 'dice_bce'   # dice_bce | dice_focal | dice_tversky
os.environ['UMUD_AUG_LEVEL'] = 'light'      # light | strong
os.environ['UMUD_FASC_POS_WEIGHT'] = '0'    # recall bias previously tested worse; keep OFF unless isolating it
# os.environ['UMUD_CLAHE'] = '1'            # train-time contrast normalization; test as its own run
# os.environ['UMUD_WEIGHTS_TAG'] = 'exp58'  # optional: save seg_apo_exp58.pt / seg_fasc_exp58.pt
print('TRAIN config -> epochs', os.environ['UMUD_EPOCHS'], '| img', os.environ['UMUD_IMG_SIZE'], '| model', os.environ['UMUD_MODEL_ARCH'], os.environ['UMUD_MODEL_ENCODER'], '| loss', os.environ['UMUD_LOSS_MODE'])
print('Inference uses fragment-extrapolation FL + scale router + inner-edge MT (all on by default).')

## EXP59 GPU matrix presets

Pick one preset, then run the full training/inference cell below. This keeps the Kaggle workflow notebook-shaped instead of relying on a shell one-liner.


In [ ]:
import os
EXP59_RUNS = {
    'seg59_01_repro_384_unet': dict(UMUD_EPOCHS='12', UMUD_IMG_SIZE='384', UMUD_MODEL_ARCH='unet', UMUD_MODEL_ENCODER='resnet34', UMUD_LOSS_MODE='dice_bce', UMUD_AUG_LEVEL='light', UMUD_CLAHE='0', UMUD_FASC_POS_WEIGHT='0', UMUD_WEIGHTS_TAG='seg59_01'),
    'seg59_02_highres_512_unet': dict(UMUD_EPOCHS='18', UMUD_IMG_SIZE='512', UMUD_MODEL_ARCH='unet', UMUD_MODEL_ENCODER='resnet34', UMUD_LOSS_MODE='dice_bce', UMUD_AUG_LEVEL='light', UMUD_CLAHE='0', UMUD_FASC_POS_WEIGHT='0', UMUD_WEIGHTS_TAG='seg59_02'),
    'seg59_03_highres_512_unetplusplus': dict(UMUD_EPOCHS='18', UMUD_IMG_SIZE='512', UMUD_MODEL_ARCH='unetplusplus', UMUD_MODEL_ENCODER='resnet34', UMUD_LOSS_MODE='dice_bce', UMUD_AUG_LEVEL='light', UMUD_CLAHE='0', UMUD_FASC_POS_WEIGHT='0', UMUD_WEIGHTS_TAG='seg59_03'),
    'seg59_04_highres_focal': dict(UMUD_EPOCHS='18', UMUD_IMG_SIZE='512', UMUD_MODEL_ARCH='unet', UMUD_MODEL_ENCODER='resnet34', UMUD_LOSS_MODE='dice_focal', UMUD_AUG_LEVEL='light', UMUD_CLAHE='0', UMUD_FASC_POS_WEIGHT='0', UMUD_WEIGHTS_TAG='seg59_04'),
    'seg59_05_train_clahe': dict(UMUD_EPOCHS='18', UMUD_IMG_SIZE='512', UMUD_MODEL_ARCH='unet', UMUD_MODEL_ENCODER='resnet34', UMUD_LOSS_MODE='dice_bce', UMUD_AUG_LEVEL='light', UMUD_CLAHE='1', UMUD_FASC_POS_WEIGHT='0', UMUD_WEIGHTS_TAG='seg59_05'),
}
RUN_ID = 'seg59_02_highres_512_unet'  # edit this line for the run to launch
for key, value in EXP59_RUNS[RUN_ID].items():
    os.environ[key] = value
print('Selected', RUN_ID)
for key in sorted(EXP59_RUNS[RUN_ID]):
    print(key, '=', os.environ[key])


## Optional smoke run (2 epochs, a few minutes)
Proves the data paths, training loop, and submission writing before committing to the full run. Skip it if you just want the full model.

In [ ]:
!UMUD_EPOCHS=2 python segment_then_measure.py

## Full run
Trains both U-Nets for `UMUD_EPOCHS` epochs (set in the config cell above — **16** for this run, with the fascicle recall bias) and overwrites `submission_segmentation.csv`. Also leaves `seg_apo.pt` and `seg_fasc.pt` in the working dir. Roughly 40-60 minutes at 16 epochs. Watch for `[fasc] using pos_weight=10.0 on BCE` and epoch lines running 0 through 15.

In [ ]:
!python segment_then_measure.py

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission_segmentation.csv')
print(sub.shape)            # expect (309, 4)
print(sub.describe())

# Sanity checks: the scale router should give per-image MT/FL, not flat constants.
assert sub.shape == (309, 4), f'expected 309x4, got {sub.shape}'
assert sub['mt_mm'].std() > 1.0, 'MT looks constant -> calibration did NOT run (scale_ticks import?)'
assert sub['fl_mm'].std() > 1.0, 'FL looks constant -> identity FL / scale did NOT run'
dbg = pd.read_csv('/kaggle/working/calibration_measurement_debug.csv')
n_scaled = int((dbg['calibration_method'] != 'none').sum())
print(f'calibrated (scaled) images: {n_scaled}/309  | methods: {dbg["calibration_method"].value_counts().to_dict()}')
assert n_scaled > 150, f'only {n_scaled} images scaled - expected ~254; check scale_ticks loaded'
print('OK: per-image MT/FL present and ~', n_scaled, 'images scaled.')

In [ ]:
# Click to download. seg_apo.pt / seg_fasc.pt are the trained weights: grab them to inspect the model locally.
import os
from IPython.display import FileLink, display
for f in ('submission_segmentation.csv', 'calibration_measurement_debug.csv', 'seg_apo.pt', 'seg_fasc.pt'):
    if os.path.exists(f):
        display(FileLink(f))